# Transformação Bronze → Silver: Customers

## Objetivo:
Transformar a tabela `bronze.bronze_customers` aplicando limpeza, padronização de tipos e implementação de lógica SCD Type 2 para gerar a tabela `silver.customers`.

## Problemas Identificados na Bronze:
- **28.813 registros** com 28.670 clientes únicos (**143 duplicados**)
- **postcode**: 45% com decimais desnecessários (.0)
- **number**: 1% alfanuméricos, resto numéricos
- **district**: Mix de códigos, siglas e nomes
- **valid_from/valid_to**: Timestamps Unix indicando versionamento
- **units_purchased**: Inteiros armazenados como DOUBLE

## Transformações Planejadas:
1. Limpeza e padronização de strings
2. Conversão de tipos de dados
3. Identificação de registros atuais (SCD Type 2)
4. Remoção de duplicados
5. Adição de colunas de auditoria
6. Validações de qualidade

---

In [0]:
# Imports necessários
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import uuid

# Configuração de variáveis
CATALOG = "retail_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
BRONZE_TABLE = "bronze_customers"
SILVER_TABLE = "customers"

# Identificador único desta execução para auditoria
pipeline_run_id = str(uuid.uuid4()) ## Onde salvamos esse pipeline_run_id? ou não salvamos ele?
processing_timestamp = datetime.now()

print(f"🔧 Configuração concluída")
print(f"📌 Pipeline Run ID: {pipeline_run_id}")
print(f"⏰ Timestamp de Processamento: {processing_timestamp}")
print(f"📥 Origem: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"📤 Destino: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

# 📖 Etapa 1: Leitura da Tabela Bronze

Carregar os dados da camada bronze e realizar uma análise inicial.

In [0]:
# Ler a tabela bronze
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

# Contagem inicial
initial_count = bronze_df.count()

print(f"✅ Tabela bronze carregada com sucesso!")
print(f"📈 Total de registros: {initial_count:,}")
print(f"\n📊 Schema da tabela bronze:")
bronze_df.printSchema()

# Exibir amostra dos dados
print(f"\n🔍 Amostra dos dados (5 primeiras linhas):")
display(bronze_df.limit(5))

# 🔍 Etapa 2: Análise de Qualidade (PRÉ-TRANSFORMAÇÃO)

Auditoria dos dados antes de aplicar transformações:
- Identificar duplicados
- Contar valores nulos
- Analisar problemas específicos em cada coluna

In [0]:
# 🔵 AUDITORIA: Contagem de duplicados por customer_id
duplicates_df = bronze_df.groupBy("customer_id").count().filter(F.col("count") > 1)
duplicates_count = duplicates_df.count()
total_duplicate_records = duplicates_df.agg(F.sum("count")).collect()[0][0]

print(f"🔴 DUPLICADOS ENCONTRADOS:")
print(f"   - {duplicates_count:,} customer_ids com duplicatas") ## total de customer_ids duplicados
print(f"   - {total_duplicate_records:,} registros duplicados no total") ## total de duplicatas
print(f"   - {initial_count - duplicates_count:,} registros únicos\n")

# 🔵 AUDITORIA: Análise de valores nulos por coluna
# Verificar se esse nullo está pegando também valores como " " que muitas vezes não conta como null
print(f"🟡 ANÁLISE DE VALORES NULOS:")
null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
])

null_summary = null_counts.collect()[0].asDict()
for col_name, null_count in sorted(null_summary.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / initial_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 🔵 AUDITORIA: Problemas específicos identificados
print(f"\n🔵 PROBLEMAS ESPECÍFICOS:")

# Postcode com decimais
postcode_with_decimal = bronze_df.filter(F.col("postcode").like("%.%")).count()
print(f"   - Postcode com .0: {postcode_with_decimal:,} ({(postcode_with_decimal/initial_count)*100:.1f}%)")

# Number com letras
number_with_letters = bronze_df.filter(F.col("number").rlike("[A-Za-z]")).count()
print(f"   - Number alfanumérico: {number_with_letters:,} ({(number_with_letters/initial_count)*100:.1f}%)")

# District numérico
district_numeric = bronze_df.filter(F.col("district").rlike("^[0-9.]+$")).count()
print(f"   - District numérico: {district_numeric:,} ({(district_numeric/initial_count)*100:.1f}%)")

# Coordenadas inválidas
invalid_coords = bronze_df.filter(
    (F.col("lon") < -180) | (F.col("lon") > 180) |
    (F.col("lat") < -90) | (F.col("lat") > 90)
).count()
print(f"   - Coordenadas inválidas: {invalid_coords:,}")

# Valid_to preenchidos (registros históricos)
valid_to_filled = bronze_df.filter(F.col("valid_to").isNotNull()).count()
print(f"   - Registros com valid_to (históricos): {valid_to_filled:,} ({(valid_to_filled/initial_count)*100:.1f}%)")

print(f"\n📊 Exemplos de registros duplicados:")
display(bronze_df.join(duplicates_df.limit(3), "customer_id", "inner").orderBy("customer_id", F.col("valid_from").desc()).limit(10))

# 🧹 Etapa 3: Limpeza e Padronização de Tipos

Aplicar transformações para:
- Limpar e padronizar strings (TRIM, UPPER)
- Remover decimais desnecessários no postcode
- Converter timestamps Unix para TIMESTAMP
- Ajustar tipos de dados (units_purchased para BIGINT)
- Validar coordenadas geográficas

In [0]:
# Aplicar todas as transformações em uma única operação
transformed_df = bronze_df.select(
    # --- IDENTIFICAÇÃO ---
    # customer_id: Manter como INT (chave primária)
    F.col("customer_id"),
    
    # tax_id e tax_code: Manter tipos originais
    F.col("tax_id"),
    F.col("tax_code"),
    
    # --- NOME E ENDEREÇO ---
    # customer_name: Limpar espaços e padronizar em maiúsculas
    F.trim(F.upper(F.col("customer_name"))).alias("customer_name"),
    
    # state: Limpar espaços e padronizar
    F.trim(F.upper(F.col("state"))).alias("state"),
    
    # city: Limpar espaços e padronizar
    F.trim(F.upper(F.col("city"))).alias("city"),
    
    # postcode: Remover .0 desnecessário e limpar
    # Usar regexp_replace para remover .0 no final + trim + upper
    F.trim(F.upper(F.regexp_replace(F.col("postcode"), "\\.0$", ""))).alias("postcode"),
    
    # street: Limpar espaços e padronizar
    F.trim(F.upper(F.col("street"))).alias("street"),
    
    # number: Manter como STRING (suporta alfanuméricos como "W333N 5591"), apenas trim
    F.trim(F.col("number")).alias("number"),
    
    # unit: Limpar espaços
    F.trim(F.col("unit")).alias("unit"),
    
    # --- REGIÃO ---
    # region: Limpar e padronizar
    F.trim(F.upper(F.col("region"))).alias("region"),
    
    # district: Limpar valores numéricos puros (que são inválidos) e padronizar
    # Se for puramente numérico (apenas dígitos e pontos), substituir por NULL
    F.when(
        F.col("district").rlike("^[0-9.]+$"), 
        None
    ).otherwise(
        F.trim(F.upper(F.col("district")))
    ).alias("district"),
    
    # --- COORDENADAS ---
    # lon: Validar range (-180 a 180), invalidar se fora do range
    F.when(
        (F.col("lon") >= -180) & (F.col("lon") <= 180),
        F.col("lon")
    ).otherwise(None).alias("lon"),
    
    # lat: Validar range (-90 a 90), invalidar se fora do range
    F.when(
        (F.col("lat") >= -90) & (F.col("lat") <= 90),
        F.col("lat")
    ).otherwise(None).alias("lat"),
    
    # ship_to_address: Limpar espaços
    F.trim(F.col("ship_to_address")).alias("ship_to_address"),
    
    # --- VERSIONAMENTO (SCD TYPE 2) ---
    # valid_from: Converter INT (Unix timestamp) para TIMESTAMP
    # Motivo: Facilita comparações de data/hora e análises temporais
    F.from_unixtime(F.col("valid_from")).cast("timestamp").alias("valid_from"),
    
    # valid_to: Converter DOUBLE (Unix timestamp) para TIMESTAMP (nullable)
    # NULL indica que o registro é a versão atual/ativa
    F.when(
        F.col("valid_to").isNotNull(),
        F.from_unixtime(F.col("valid_to")).cast("timestamp")
    ).alias("valid_to"),
    
    # --- MÉTRICAS ---
    # units_purchased: Converter de DOUBLE para BIGINT
    # Motivo: Todos os valores são inteiros, não há decimais
    F.col("units_purchased").cast("bigint").alias("units_purchased"),
    
    # loyalty_segment: Manter INT (valores de 0 a 3)
    F.col("loyalty_segment")
)

print("✅ Transformações aplicadas com sucesso!")
print(f"\n📊 Novo schema após transformações:")
transformed_df.printSchema()

print(f"\n🔍 Amostra dos dados transformados (5 primeiras linhas):")
display(transformed_df.limit(5))

# 🎯 Etapa 4: Identificação de Registros Mais Recentes (SCD Type 2)

## O que é SCD Type 2?
Slowly Changing Dimension Type 2 é uma técnica para manter o **histórico de mudanças** nos dados.

## Como funciona?
- Cada customer_id pode ter **múltiplas versões** ao longo do tempo
- **valid_from**: Data/hora em que a versão começou a ser válida
- **valid_to**: Data/hora em que a versão deixou de ser válida (NULL = versão atual)
- **is_current**: Flag booleana indicando se é a versão mais recente

## Lógica de Identificação:
1. Ordenar registros de cada customer_id por **valid_from DESC**
2. O registro com **valid_from** mais recente é a versão atual
3. Adicionar coluna **is_current** e **row_number** para auditoria

In [0]:
# Definir window para particionar por customer_id e ordenar por valid_from DESC
# Isso permite identificar qual é o registro mais recente de cada cliente
window_spec = Window.partitionBy("customer_id").orderBy(F.col("valid_from").desc())

# Adicionar colunas de controle SCD Type 2
scd_df = transformed_df.withColumn(
    # row_number: Numeração dos registros (1 = mais recente, 2 = segundo mais recente, etc.)
    "row_number",
    F.row_number().over(window_spec)
).withColumn(
    # is_current: Flag booleana indicando se é a versão atual
    # TRUE se:
    #   - row_number = 1 (registro mais recente por valid_from)
    #   - OU valid_to IS NULL (indica versão ativa)
    #   - OU valid_to > timestamp atual (ainda não expirou)
    "is_current",
    F.when(
        (F.col("row_number") == 1) | 
        (F.col("valid_to").isNull()) |
        (F.col("valid_to") > F.current_timestamp()),
        True
    ).otherwise(False)
)

# 🔵 AUDITORIA: Análise da distribuição de versões
print("🔵 AUDITORIA SCD TYPE 2:")
print(f"\n📊 Distribuição de versões por customer_id:")

version_distribution = scd_df.groupBy("customer_id").agg(
    F.count("*").alias("num_versions")
).groupBy("num_versions").count().orderBy("num_versions")

display(version_distribution)

# Contar registros atuais vs históricos
current_count = scd_df.filter(F.col("is_current") == True).count()
historical_count = scd_df.filter(F.col("is_current") == False).count()

print(f"\n✅ Registros ATUAIS (is_current = TRUE): {current_count:,}")
print(f"📋 Registros HISTÓRICOS (is_current = FALSE): {historical_count:,}")
print(f"📈 Total: {current_count + historical_count:,}")

# Exemplos de clientes com múltiplas versões
print(f"\n🔍 Exemplo: Cliente com múltiplas versões (histórico de mudanças):")
multi_version_customer = scd_df.filter(F.col("row_number") > 1).select("customer_id").limit(1).collect()

if multi_version_customer:
    example_customer_id = multi_version_customer[0][0]
    display(
        scd_df.filter(F.col("customer_id") == example_customer_id)
        .select("customer_id", "customer_name", "city", "state", "valid_from", "valid_to", "is_current", "row_number")
        .orderBy("valid_from")
    )

# 🗑️ Etapa 5: Remoção de Duplicados

Manter apenas o **registro mais recente** de cada customer_id:
- Filtrar por `row_number = 1` (mais recente por valid_from)
- Remover a coluna auxiliar `row_number` após filtro
- **Garantir** que cada customer_id apareça apenas uma vez na Silver

In [0]:
# Filtrar apenas registros mais recentes (row_number = 1)
# Isso garante que cada customer_id tenha apenas 1 registro na camada Silver
silver_df = scd_df.filter(F.col("row_number") == 1).drop("row_number")

# Contagem após remoção de duplicados
silver_count = silver_df.count()
removed_duplicates = initial_count - silver_count

print("✅ Duplicados removidos com sucesso!")
print(f"\n🔵 AUDITORIA - REMOÇÃO DE DUPLICADOS:")
print(f"   - Registros na Bronze: {initial_count:,}")
print(f"   - Registros na Silver: {silver_count:,}")
print(f"   - Duplicados removidos: {removed_duplicates:,}")
print(f"   - Redução: {(removed_duplicates/initial_count)*100:.2f}%")

# Verificar se customer_id é único
unique_customers = silver_df.select("customer_id").distinct().count()
print(f"\n✅ Verificação de unicidade:")
print(f"   - Total de registros: {silver_count:,}")
print(f"   - Customer IDs únicos: {unique_customers:,}")
if silver_count == unique_customers:
    print(f"   - ✅ SUCESSO: Cada customer_id é único na Silver!")
else:
    print(f"   - ⚠️ ALERTA: Ainda existem duplicados!")

# ✅ Etapa 6: Validações Finais (PÓS-TRANSFORMAÇÃO)

Verificar a qualidade dos dados transformados:
- Confirmar unicidade de customer_id
- Analisar valores nulos após limpeza
- Validar ranges de coordenadas
- Comparar estatísticas antes/depois

In [0]:
print("🔵 AUDITORIA - VALIDAÇÕES PÓS-TRANSFORMAÇÃO")
print("="*60)

# 1. Confirmar unicidade de customer_id
print(f"\n1️⃣ UNICIDADE DE CUSTOMER_ID:")
duplicates_check = silver_df.groupBy("customer_id").count().filter(F.col("count") > 1).count()
if duplicates_check == 0:
    print(f"   ✅ Customer_id é único (0 duplicados)")
else:
    print(f"   ⚠️ {duplicates_check} customer_ids ainda estão duplicados!")

# 2. Análise de valores nulos após limpeza
print(f"\n2️⃣ VALORES NULOS APÓS LIMPEZA:")
null_counts_after = silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in silver_df.columns if c not in ["row_number"]
])

null_summary_after = null_counts_after.collect()[0].asDict()
for col_name, null_count in sorted(null_summary_after.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / silver_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 3. Validar coordenadas
print(f"\n3️⃣ VALIDAÇÃO DE COORDENADAS:")
valid_coords = silver_df.filter(
    F.col("lon").isNotNull() & F.col("lat").isNotNull()
).count()
invalid_coords_after = silver_df.filter(
    F.col("lon").isNull() | F.col("lat").isNull()
).count()
print(f"   - Coordenadas válidas: {valid_coords:,} ({(valid_coords/silver_count)*100:.1f}%)")
print(f"   - Coordenadas ausentes/inválidas: {invalid_coords_after:,} ({(invalid_coords_after/silver_count)*100:.1f}%)")

# 4. Estatísticas de is_current
print(f"\n4️⃣ DISTRIBUIÇÃO DE REGISTROS ATUAIS:")
is_current_stats = silver_df.groupBy("is_current").count().collect()
for row in is_current_stats:
    status = "ATUAL" if row["is_current"] else "HISTÓRICO"
    count = row["count"]
    print(f"   - {status}: {count:,} registros ({(count/silver_count)*100:.1f}%)")

# 5. Estatísticas de units_purchased
print(f"\n5️⃣ ESTATÍSTICAS DE UNITS_PURCHASED:")
units_stats = silver_df.select(
    F.min("units_purchased").alias("min"),
    F.max("units_purchased").alias("max"),
    F.avg("units_purchased").alias("avg"),
    F.sum("units_purchased").alias("total")
).collect()[0]

print(f"   - Mínimo: {units_stats['min']}")
print(f"   - Máximo: {units_stats['max']}")
print(f"   - Média: {units_stats['avg']:.2f}")
print(f"   - Total: {units_stats['total']:,}")

print(f"\n" + "="*60)
print("✅ Validações concluídas!")

# 📝 Etapa 7: Adição de Colunas de Auditoria

Adicionar metadados para rastreabilidade:
- **processed_at**: Timestamp de quando foi processado
- **source_table**: Tabela de origem
- **pipeline_run_id**: Identificador único desta execução
- **data_quality_score**: Métrica de qualidade (% de campos não-nulos)

In [0]:
# Calcular data quality score para cada registro
# Score = (número de campos não-nulos / total de campos) * 100
# Excluindo campos de auditoria e controle

data_columns = [
    "customer_id", "tax_id", "tax_code", "customer_name", "state", "city",
    "postcode", "street", "number", "unit", "region", "district",
    "lon", "lat", "ship_to_address", "units_purchased", "loyalty_segment"
]

total_fields = len(data_columns)

# Contar campos não-nulos para cada registro
non_null_count = sum([
    F.when(F.col(c).isNotNull(), 1).otherwise(0) 
    for c in data_columns
])

# Adicionar colunas de auditoria
final_df = silver_df.withColumn(
    "data_quality_score",
    F.round((non_null_count / total_fields) * 100, 2)
).withColumn(
    "processed_at",
    F.lit(processing_timestamp).cast("timestamp")
).withColumn(
    "source_table",
    F.lit(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
).withColumn(
    "pipeline_run_id",
    F.lit(pipeline_run_id)
)

print("✅ Colunas de auditoria adicionadas!")
print(f"\n📊 Schema final com metadados:")
final_df.printSchema()

print(f"\n🔵 AUDITORIA - DATA QUALITY SCORE:")

# 🔧 CORREÇÃO: Adicionar alias para a expressão do groupBy
quality_distribution = final_df.groupBy(
    (F.floor(F.col("data_quality_score") / 10) * 10).alias("quality_bucket")
).count().orderBy("quality_bucket")

print(f"\n📈 Distribuição de qualidade dos dados:")
display(quality_distribution)

avg_quality = final_df.agg(F.avg("data_quality_score")).collect()[0][0]
print(f"\n🎯 Score médio de qualidade: {avg_quality:.2f}%")

print(f"\n🔍 Amostra dos dados finais com auditoria:")
display(final_df.limit(5))

# 📦 Etapa 8: Escrita na Tabela Silver

Persistir os dados transformados na camada Silver:
- **Formato**: Delta Lake (suporte a ACID, Time Travel, Change Data Feed)
- **Modo**: Overwrite (primeira carga) / Merge (cargas incrementais)
- **Partição**: Opcional por estado para melhor performance
- **Recursos Delta**: Habilitar Change Data Feed e otimizações

In [0]:
# Iniciar timer para medir tempo de escrita
import time
write_start = time.time()

# Escrever na tabela Silver com configurações Delta Lake
silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"🔄 Iniciando escrita na tabela Silver...")
print(f"📍 Destino: {silver_table_name}")

# Escrever com Delta Lake
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(silver_table_name)

write_end = time.time()
write_duration = write_end - write_start

print(f"\n✅ Tabela Silver criada com sucesso!")
print(f"⏱️ Tempo de escrita: {write_duration:.2f} segundos")

# Otimizar a tabela (compactação de arquivos pequenos)
print(f"\n🛠️ Otimizando tabela...")
spark.sql(f"OPTIMIZE {silver_table_name}")
print("✅ Otimização concluída!")

# Coletar estatísticas da tabela
print(f"\n📊 Coletando estatísticas da tabela...")
spark.sql(f"ANALYZE TABLE {silver_table_name} COMPUTE STATISTICS")
print("✅ Estatísticas coletadas!")

# Verificar a tabela criada
print(f"\n🔍 Verificação da tabela Silver:")
silver_verification = spark.sql(f"SELECT COUNT(*) as count FROM {silver_table_name}").collect()[0][0]
print(f"   - Total de registros: {silver_verification:,}")

if silver_verification == silver_count:
    print(f"   ✅ Verificação bem-sucedida! Todos os {silver_count:,} registros foram gravados.")
else:
    print(f"   ⚠️ Divergência! Esperado: {silver_count:,}, Encontrado: {silver_verification:,}")

# 📊 Etapa 9: Relatório Final de Transformação

Resumo completo de todas as transformações aplicadas e métricas de auditoria.